# Projeto de Controladores

In [225]:
# Bibliotecas utilizadas
import matplotlib.pyplot as plt
import sympy as sp
from sympy import symbols, I, Abs, re, im, Poly
from sympy.physics.control import *
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath
from collections import Counter
#import plotly.graph_objects as go

In [226]:
# Símbolos utilizados
s, k = sp.symbols('s k')

## Funções auxiliares

In [227]:
def polinomio(entrada):
    '''
    Função que transforma coeficientes em polinômios
    '''
    coeficientes = entrada.strip().split()  # Transforma a entrada em uma lista de strings e remove espaços extras
    coeficientes.reverse() # Inverte a lista
    lista_polinomio = []

    # Percorre de trás para frente para começar pelo maior expoente
    for exp in range(len(coeficientes) - 1, -1, -1):
        coef = coeficientes[exp]

        # Pula coeficientes que são zero
        if coef == "0":
            continue

        # Formata o termo dependendo do expoente
        if exp > 1:
            termo = f"{coef}*s**{exp}"
        elif exp == 1:
            termo = f"{coef}*s"
        else:
            termo = f"{coef}"

        lista_polinomio.append(termo)

    return " + ".join(lista_polinomio)


In [228]:
def definir_controlador():
    PI = False
    PD = False
    PID = False
    controlador = ""

    while(controlador != "PI" and controlador != "PD" and controlador != "PID"):
        controlador = input("Forneça o tipo de controlador").upper()
        if(controlador == "PI"):
            PI = True
            print("Controlador PI")
        elif(controlador == "PD"):
            PD = True
            print("Controlador PD")
        elif(controlador == "PID"):
            PID = True
            print("Controlador PID")
    return PI,PD,PID


In [229]:
def definir_problema():
    metodo_1 = False
    metodo_2 = False
    metodo_3 = False
    print("Digite 1 para Mp e Epsilon.")
    print("Digite 2 para fator de amortecimento e frequência natural de amortecimento.")
    print("Digite 3 para localização de polos de malha")
    especificacoes = 0
    while(especificacoes < 1 or especificacoes > 3):
        especificacoes = int(input("Digite o número: "))
        if(especificacoes == 1):
            metodo_1 = True
        elif(especificacoes == 2):
            metodo_2 = True
        elif(especificacoes == 3):
            metodo_3 = True
    return metodo_1, metodo_2, metodo_3


In [230]:
def definir_polos_dominantes(metodo1, metodo2, metodo3):
    if(metodo1):
        Mp = float(input("Digite o Mp máximo (%): "))
        print(f"Mp = {Mp}%")
        t_acomodacao = float(input("Digite o tempo de acomodação (s):"))
        tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        while(tol != 2 and tol != 5):
            tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        
        print(f"Acomodação {t_acomodacao}s ({tol}%)")

        # Cálculo do epsilon e frequência natural
        epsilon = math.sqrt(math.log(Mp/100)**2/(math.pi**2 + math.log(Mp/100)**2))
        if(tol == 2):
            w_n = 4/(t_acomodacao*epsilon)
        elif(tol == 5):
            w_n = 3/(t_acomodacao*epsilon)

        w_d = w_n*math.sqrt(1 - epsilon**2)
        sigma = epsilon*w_n

        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d  * I

    if(metodo2):
        epsilon = float(input("Digite o valor do epsilon: "))
        w_n = float(input("Digite o valor da frequência natural de oscilação: "))
        w_d = w_n*math.sqrt(1 - epsilon**2)
        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d * I
    if(metodo3):
        parte_real = float(input("Digite a parte real do polo: "))
        parte_imaginaria = float(input("Digite a parte imaginária do polo: "))
        w_d = parte_imaginaria
        sigma = - parte_real
        polo_1 = complex(parte_real, parte_imaginaria)
        polo_2 = complex(parte_real, - parte_imaginaria)

    return polo_1, polo_2, w_d, sigma


In [231]:
def angulo(v):
    """Retorna o argumento de um número complexo em graus."""
    return float(sp.deg(sp.arg(v)))


In [244]:
def calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, pid):
    ponto_si = polo_1

    diferenca_polos = np.empty((0,2))
    diferenca_zeros = np.empty((0,2))

    #print("Diferença entre o ponto e os polos: ")
    for polo in polos:
        diferenca = ponto_si - polo
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[round(modulo,4), fase]]
        diferenca_polos = np.append(diferenca_polos, nova_linha, axis=0)
        print(f"{polo}: {nova_linha}")
    #print

    #print("Diferença entre o ponto e os zeros: ")
    for zero in zeros:
        diferenca = ponto_si - zero
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[round(modulo,4), fase]]
        diferenca_zeros = np.append(diferenca_zeros, nova_linha, axis=0)
        print(f"{zero}: {nova_linha}")

    produto_polos = 1
    produto_zeros = 1
    angulo_ponto = 0

    for linha in diferenca_polos:
        produto_polos = produto_polos * linha[0]
        angulo_ponto = angulo_ponto - linha[1]

    for linha in diferenca_zeros:
        produto_zeros = produto_zeros * linha[0]
        angulo_ponto = angulo_ponto + linha[1]

    phi = 180 - angulo_ponto

    while phi < 0:
        phi += 360
        
    phi = phi % 360

    if(PID):
        phi = phi /2

    print(f"Angulo = {phi:.2f}°")

    if phi > 90:
        z_c = abs(sigma) - (w_d / math.tan(math.radians(180 - phi)))
    else:
        z_c = abs(sigma) + (w_d / math.tan(math.radians(phi)))
    print(f"Zero do controlador: {z_c:.2f} + 0*j")

    if(PID):
        Gc = TransferFunction((s + z_c)**2, 1, s)
    else:
        Gc = TransferFunction(s + z_c, 1, s)
    ft_c = Series(ft, Gc).doit()

    freq = ft.eval_frequency(ponto_si)

    Gc_sem_K = (ponto_si + z_c)**2

    M_G = abs(complex(freq))
    M_Gc = abs(complex(Gc_sem_K))

    Kc = 1 / (M_G * M_Gc)

    print(f"|G(sd)|       = {M_G}")
    print(f"|Gc(sd)|      = {M_Gc}")
    print(f"|G(sd)Gc(sd)| = {M_G * M_Gc}")
    print(f"Kc            = {Kc}")
    print(f"Kc = {Kc}\n")
    
    return z_c, Kc

## Traduzir as especificações de desempenho

In [233]:
PI, PD, PID = definir_controlador()

Controlador PID


In [234]:
metodo_1, metodo_2, metodo_3 = definir_problema()

Digite 1 para Mp e Epsilon.
Digite 2 para fator de amortecimento e frequência natural de amortecimento.
Digite 3 para localização de polos de malha


In [235]:
polo_1, polo_2, w_d, sigma = definir_polos_dominantes(metodo_1, metodo_2, metodo_3)
print(polo_1, polo_2)

(-1+1.73j) (-1-1.73j)


In [236]:
print("Forneça os coeficientes da função G(s)...")
numerador_G = polinomio(input("Digite os coeficientes do numerador: "))
denominador_G = polinomio(input("Digite os coeficientes do denominador: "))

numerador_G = sp.parse_expr(numerador_G, local_dict={'s': s})
denominador_G = sp.parse_expr(denominador_G, local_dict={'s': s})

G = TransferFunction(numerador_G, denominador_G, s)
G

Forneça os coeficientes da função G(s)...


TransferFunction(1, s**2 + 1, s)

In [237]:
print("Forneça os coeficientes da função H(s)...")
numerador_H = polinomio(input("Digite os coeficientes do numerador: "))
denominador_H = polinomio(input("Digite os coeficientes do denominador: "))

numerador_H = sp.parse_expr(numerador_H, local_dict={'s': s})
denominador_H = sp.parse_expr(denominador_H, local_dict={'s': s})

H = TransferFunction(numerador_H, denominador_H, s)
H

Forneça os coeficientes da função H(s)...


TransferFunction(1, 1, s)

In [238]:
ft = Series(G, H).doit()
ft

TransferFunction(1, s**2 + 1, s)

In [239]:
zeros, polos = pole_zero_numerical_data(ft)
print(f"Polos: {polos}")
print(f"Zeros: {zeros}")

Polos: [-1j, 1j]
Zeros: []


In [240]:
# Verifica se a função de transferência tem naturalmente um ganho
num = ft.num
den = ft.den

ganho = Poly(num, s).LC() / Poly(den, s).LC()
print(f"Ganho = {ganho}")

Ganho = 1


## PD

In [241]:
if(PD):
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kd = Kc
    Kp = z_c * Kc
    print(f"Kp = {Kp} \nKd = {Kd}")
    

## PI

In [242]:
if(PI):
    zeros, polos = pole_zero_numerical_data(ft) # para ao executar novamente não mudar os resultados
    polos.append(0 + 0*I)
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kp = Kc
    Ki = z_c * Kc
    print(f"Kp = {Kp} \nKi = {Ki}")
    

In [245]:
if(PID):
    zeros, polos = pole_zero_numerical_data(ft)  # para ao executar novamente não mudar os resultados
    polos.append(0 + 0*I)
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kp = 2*Kc*z_c
    Ki = Kc * z_c**2
    Kd = Kc
    print(f"Kp = {Kp} \nKi = {Ki} \nKd = {Kd}")
    

-1j: [[2.9074, 110.11780102431226]]
1j: [[1.2381, 143.87055585675915]]
0: [[1.9982, 120.02940176151468]]
Angulo = 97.01°
Zero do controlador: 0.79 + 0*j
|G(sd)|       = 0.2778051050909076
|Gc(sd)|      = 3.038136798508219
|G(sd)Gc(sd)| = 0.8440099125901294
Kc            = 1.1848202077759518
Kc = 1.1848202077759518

Kp = 1.8656429191090642 
Ki = 0.7344201843407394 
Kd = 1.1848202077759518
